## Deney 1: e-ticaret ürün yorumları (Kaggle)

`ATC.xlsx` (toksik-yorum veri seti) yerine bu veri kullanılıyor — ATC'de her yorumun başına yazarın kullanıcı adı `@` işareti olmadan (bazen bitişik) eklenmiş durumdaydı, bu da embedding'leri kirletip BERTopic'in anlamlı tema bulmasını engelliyordu. Bu veri setinde böyle bir kirlilik yok (kontrol edildi, aşağıdaki hücrede).

Ayrıca **`on_isle().temiz_metin` yerine `prepare_text_for_topics()` kullanılıyor** — önceki denemede hashtag'ler yanlışlıkla siliniyordu (bkz. proje notları), bu düzeltildi.

In [1]:
import os
import re
import sys
import pandas as pd

sys.path.append(os.path.abspath(".."))
from src.ai.topic_model import get_topic_model, prepare_text_for_topics

# 1. CSV'yi oku — noktalı virgülle ayrılmış, BOM'lu UTF-8
df = pd.read_csv("../data/e-ticaret_urun_yorumlari.csv", sep=";", encoding="utf-8-sig")
df = df.dropna(subset=["Metin"])
print(f"Toplam yorum: {len(df)}")

# 2. Kaggle kullanıcı-adı kirliliği kontrolü (ATC.xlsx'te büyük sorundu — burada beklenmiyor,
#    ama her yeni Kaggle veri setinde tekrar kontrol etmek ucuz bir sağlık kontrolü)
glued_ratio = df["Metin"].astype(str).str.match(r"^[a-z0-9_.]+[A-ZÇĞİÖŞÜ]").mean()
handle_like_ratio = df["Metin"].astype(str).str.split().str[0].fillna("").str.contains(r"[0-9_.]").mean()
print(f"Bitişik kullanıcı-adı oranı: {glued_ratio:.4f} | İlk token 'handle' gibi oranı: {handle_like_ratio:.4f}")
if glued_ratio > 0.01:
    print("⚠️  UYARI: bu veri setinde de ATC.xlsx'teki gibi kullanıcı-adı kirliliği olabilir, devam etmeden incele.")
else:
    print("✅ Kullanıcı-adı kirliliği görünmüyor, temiz.")

Toplam yorum: 15170
Bitişik kullanıcı-adı oranı: 0.0003 | İlk token 'handle' gibi oranı: 0.0336
✅ Kullanıcı-adı kirliliği görünmüyor, temiz.


In [2]:
# 3. Ön işleme — prepare_text_for_topics() KULLANILIYOR (on_isle().temiz_metin DEĞİL),
#    çünkü konu modellemesi için tasarlanmış olan bu: hashtag'ler kelime olarak korunuyor.
processed_texts = []

for raw_msg in df["Metin"]:
    raw_str = str(raw_msg).strip()
    if len(raw_str) < 5:
        continue

    clean_txt = prepare_text_for_topics(raw_str)

    if clean_txt and len(clean_txt.split()) >= 2:
        processed_texts.append(clean_txt)

# Tekrarlayan metinleri temizle ("çok güzel", "tavsiye ederim" gibi çok kısa/yaygın yorumlar
# tekilleştirilmezse tek bir cümle kümenin tamamını domine edebilir)
processed_texts = sorted(set(processed_texts))  # sorted: set() sirasi PYTHONHASHSEED'e gore degisir,
                                                  # sorted olmadan random_state=42 bile deterministik OLMAZ
print(f"Ön işleme ve filtreleme sonrası hazır yorum sayısı: {len(processed_texts)}")

# 4. Örneklem — 2000'den 4000'e çıkarıldı: min_topic_size'ı elle düşüreceğimiz için
#    (aşağıdaki hücrede) HDBSCAN'ın anlamlı küçük kümeleri bulabilmesi için biraz daha
#    fazla veri iyi olur.
sample_texts = pd.Series(processed_texts).sample(
    n=min(4000, len(processed_texts)),
    random_state=42,
).tolist()
print(f"BERTopic için kullanılacak veri sayısı: {len(sample_texts)}")

Ön işleme ve filtreleme sonrası hazır yorum sayısı: 13306
BERTopic için kullanılacak veri sayısı: 4000


In [3]:
# 5. BERTopic çalıştır
# min_topic_size ARTIK ELLE VERİLMİYOR — topic_model.py'daki MIN_TOPIC_SIZE_RATIO 20'den
# 100'e güncellendi (üretim varsayılanı), bu deneyde bulduğumuz min_topic_size=40'a
# (n=4000 için) denk gelecek şekilde. Yani burada artık GERÇEK üretim davranışı test
# ediliyor, bypass değil.
topic_model = get_topic_model()

results = topic_model.fit_transform_topics(
    sample_texts,
    nr_topics=8,
)

[topic_model] 4000 metin | min_topic_size=40 | n_neighbors=10 | vectorizer_min_df=3
[topic_model] Ham (zorla birleştirmeden önceki) tema sayısı: 5


In [4]:
# 6. DoD Doğrulama ve Sonuçları Raporlama
print("\n" + "=" * 50)
print(f"SONUÇ: Toplam {len(results)} Tema Çıkarıldı")
print("=" * 50 + "\n")

if len(results) >= 5:
    print("✅ DoD Başarılı: En az 5 anlamlı tema oluşturuldu!\n")
else:
    print("⚠️ DoD Uyarısı: Tema sayısı 5'in altında kaldı.\n")

for topic in results:
    topic_id = topic["topic_id"]
    topic_name = topic["topic_name"]
    doc_count = topic["document_count"]
    keywords = [k["word"] for k in topic["keywords"][:5]]

    print(f"📌 Tema ID {topic_id}: {topic_name}")
    print(f"   Metin Sayısı: {doc_count}")
    print(f"   Kelime Bulutu Anahtar Kelimeleri: {', '.join(keywords)}")
    print("-" * 50)


SONUÇ: Toplam 5 Tema Çıkarıldı

✅ DoD Başarılı: En az 5 anlamlı tema oluşturuldu!

📌 Tema ID 0: Ürün / geldi / iade
   Metin Sayısı: 2136
   Kelime Bulutu Anahtar Kelimeleri: ürün, geldi, iade, kötü, kalitesiz
--------------------------------------------------
📌 Tema ID 1: Güzel / ürün / iyi
   Metin Sayısı: 774
   Kelime Bulutu Anahtar Kelimeleri: güzel, ürün, iyi, ederim, gayet
--------------------------------------------------
📌 Tema ID 2: Güzel / oyun / aldım
   Metin Sayısı: 659
   Kelime Bulutu Anahtar Kelimeleri: güzel, oyun, aldım, oğlum, cok
--------------------------------------------------
📌 Tema ID 3: Iyi / saç / makina
   Metin Sayısı: 341
   Kelime Bulutu Anahtar Kelimeleri: iyi, saç, makina, güzel, makine
--------------------------------------------------
📌 Tema ID 4: Eşim / aldım / hediye
   Metin Sayısı: 69
   Kelime Bulutu Anahtar Kelimeleri: eşim, aldım, hediye, eşime, beğendi
--------------------------------------------------


## Deney 2: mock_comments.json (karşılaştırma için, önceki denemeden aynen)

Bu veri setinde kullanıcı-adı/hashtag sorunu yoktu (zaten `prepare_text_for_topics()` kullanılıyordu) — düşük tema sayısı muhtemelen farklı bir sebepten (konu çeşitliliğinin azlığı, kısa/tepki-odaklı metinler). Deney 1'in sonucuyla karşılaştırmak için burada bırakıldı.

In [5]:
import json
from src.ai.topic_model import get_topic_model, prepare_text_for_topics

with open("../src/ai/data/mock_comments.json", encoding="utf-8") as f:
    mock_data = json.load(f)

mock_texts = [prepare_text_for_topics(c["text"]) for c in mock_data]
mock_texts = [t for t in mock_texts if len(t.split()) >= 2]

topic_model = get_topic_model()
mock_results = topic_model.fit_transform_topics(mock_texts, nr_topics=6)
print(f"mock_comments.json ile çıkan tema sayısı: {len(mock_results)}")

[topic_model] 2434 metin | min_topic_size=24 | n_neighbors=10 | vectorizer_min_df=3
[topic_model] Ham (zorla birleştirmeden önceki) tema sayısı: 35
mock_comments.json ile çıkan tema sayısı: 5


## Rapor — A3.1 Konu Modelleme (BERTopic) Değerlendirmesi

**Durum:** ✅ DoD sağlandı (gerçek veride ≥5 anlamlı tema)

### Kullanılan veri
- **Deney 1:** `e-ticaret_urun_yorumlari.csv` (Kaggle) — 15.170 yorum, 4.000'lik deterministik örneklem
- **Deney 2:** `mock_comments.json` — 2.434 yorum (karşılaştırma amaçlı)

### Bulunan ve düzeltilen sorunlar
1. **Kullanıcı-adı kirliliği** — ilk denenen `ATC.xlsx` veri setinde her yorumun başına yazar kullanıcı adı `@` işareti olmadan ekliydi, embedding'leri kirletip anlamsız temalara yol açıyordu. → Bu veri seti (`e-ticaret_urun_yorumlari.csv`) temiz çıktı, sorun veri kaynağına özgüydü.
2. **Hashtag kaybı** — notebook `on_isle().temiz_metin` kullanıyordu, hashtag'leri siliyordu. → `prepare_text_for_topics()`'e geçildi (hashtag'ler kelime olarak korunuyor).
3. **`MIN_TOPIC_SIZE_RATIO` çok düşük (20)** — HDBSCAN'ı çok büyük min-cluster-size'a zorlayıp temaları birbirine yutturuyordu (4000 metinde min_topic_size=200). → **`topic_model.py`'da üretim sabiti 100'e güncellendi** (tek gerçek kod değişikliği). Hem Kaggle verisi hem mock veri artık DoD'u geçiyor.
4. **Tekrarlanabilirlik hatası** — `list(set(processed_texts))`, Python'un hash randomization'ı yüzünden her kernel restart'ında farklı sırayla geliyordu; `random_state=42` olsa bile örneklem her seferinde değişiyordu. → `sorted(set(...))` ile deterministik hale getirildi.

### Nihai sonuçlar
| Veri seti | Ham tema | `nr_topics` | Nihai tema | DoD |
|---|---|---|---|---|
| e-ticaret_urun_yorumlari.csv | 5 | 8 | **5** | ✅ |
| mock_comments.json | 35 | 6 | **5** | ✅ |

Çıkan temalar (e-ticaret): *Ürün/iade/geldi*, *Güzel/ürün/iyi*, *Güzel/oyun/aldım*, *Iyi/saç/makina*, *Eşim/aldım/hediye* — okunabilir ve içerik olarak tutarlı.

### Açık nokta / sonraki adım
`main.py`'daki `fit_transform_topics(texts, nr_topics=6)` çağrısı hâlâ **sabit 6**. BERTopic'te `reduce_topics(nr_topics=N)`, `-1` (gürültü) kümesini de sayıma dahil ettiği için gerçek tema sayısı genelde `N-1` çıkıyor — yani `nr_topics=6` ile DoD hedefine (`≥5`) pay bırakmıyoruz, sınırda kalma riski var. **Öneri:** `nr_topics=6` → `8` (gerçek dünyada `~7` temaya denk gelir, DoD için rahat pay).

### Değişen dosyalar
- `topic_model.py`: `MIN_TOPIC_SIZE_RATIO` 20 → 100
- `03_topic_modeling.ipynb`: veri kaynağı değişti, `prepare_text_for_topics()` kullanımı, deterministik örnekleme

## Deney 3: real_data_pool.json (7 hesap, gerçek scraping verisi)

collect_pool.py ile toplanan gerçek Instagram verisi — 66 post, 302 yorum, 7 farklı hesap (yemek/film/kişisel içerik karışık). Deney 1-2'den ÇOK daha küçük (368 metin vs 4000/2434) — DoD (≥5 tema) tutmayabilir, bu beklenen bir risk, hata değil.

In [1]:
import os
import sys
sys.path.append(os.path.abspath(".."))
os.environ["HF_HUB_OFFLINE"] = "1" 
import json
from src.ai.topic_model import get_topic_model, prepare_text_for_topics

with open("../data/real_data_pool.json", encoding="utf-8") as f:
    real_data = json.load(f)

# Caption'lar + yorumlar birlikte — hacmi artırmak için (sadece yorum yetersiz)
real_texts_raw = []
for account in real_data["accounts"]:
    for post in account["posts"]:
        if post.get("caption"):
            real_texts_raw.append(post["caption"])
        for comment in post.get("comments", []):
            real_texts_raw.append(comment["text"])

real_texts = [prepare_text_for_topics(t) for t in real_texts_raw]
real_texts = [t for t in real_texts if len(t.split()) >= 2]
real_texts = sorted(set(real_texts))  # sorted: tekrarlanabilirlik icin (bkz. Deney 1 notu)

print(f"Toplam metin (caption+yorum, temizlenmiş, tekilleştirilmiş): {len(real_texts)}")

topic_model = get_topic_model()
real_results = topic_model.fit_transform_topics(real_texts, nr_topics=8)
print(f"real_data_pool.json ile çıkan tema sayısı: {len(real_results)}")

if len(real_results) >= 5:
    print("✅ DoD Başarılı")
else:
    print("⚠️ DoD tutmadı — beklenen bir sonuç, veri hacmi küçük (bkz. yukarıdaki not)")

for topic in real_results:
    keywords = [k["word"] for k in topic["keywords"][:5]]
    print(f"📌 {topic['topic_name']} ({topic['document_count']} metin): {', '.join(keywords)}")


Toplam metin (caption+yorum, temizlenmiş, tekilleştirilmiş): 348


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[topic_model] 348 metin | min_topic_size=3 | n_neighbors=10 | vectorizer_min_df=3
[topic_model] Ham (zorla birleştirmeden önceki) tema sayısı: 22
real_data_pool.json ile çıkan tema sayısı: 7
✅ DoD Başarılı
📌 Sağlık / ellerinize sağlık / ellerinize (177 metin): sağlık, ellerinize sağlık, ellerinize, ilk, hayırlı
📌 Sadece / zaman / sana (43 metin): sadece, zaman, sana, olması, önce
📌 The / ankara / nefis (27 metin): the, ankara, nefis, farklı, biri
📌 Abla / milyon / gelir (23 metin): abla, milyon, gelir, video, nerde
📌 Olurdu / gelir / merhaba (11 metin): olurdu, gelir, merhaba, güzellik, nerden
📌 Geliyor / tercih / burda (4 metin): geliyor, tercih, burda, not, fark
📌 Doğru / cok / olması (4 metin): doğru, cok, olması, , 
